In [3]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

Using device: cuda


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("gowrishankarp/newspaper-text-summarization-cnn-dailymail")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/newspaper-text-summarization-cnn-dailymail


In [5]:
import pandas as pd

train_df = pd.read_csv(path+ "/cnn_dailymail/train.csv")
test_df = pd.read_csv(path+ "/cnn_dailymail/test.csv")
validation_df = pd.read_csv(path+ "/cnn_dailymail/validation.csv")

In [6]:
train_df

,id,article,highlights
0,0001d1afc246a7964130f43ae940af6bc6c57f01,By . Associated Press . PUBLISHED: . 14:11 EST...,"Bishop John Folda, of North Dakota, is taking ..."
1,0002095e55fcbd3a2f366d9bf92a95433dc305ef,(CNN) -- Ralph Mata was an internal affairs li...,Criminal complaint: Cop used his role to help ...
2,00027e965c8264c35cc1bc55556db388da82b07f,A drunk driver who killed a young woman in a h...,"Craig Eccleston-Todd, 27, had drunk at least t..."
3,0002c17436637c4fe1837c935c04de47adb18e9a,(CNN) -- With a breezy sweep of his pen Presid...,Nina dos Santos says Europe must be ready to a...
4,0003ad6ef0c37534f80b55b4235108024b407f0b,Fleetwood are the only team still to have a 10...,Fleetwood top of League One after 2-0 win at S...
...,...,...,...
287108,fffdfb56fdf1a12d364562cc2b9b1d4de7481dee,By . James Rush . Former first daughter Chelse...,Chelsea Clinton said question of running for o...
287109,fffeecb8690b85de8c3faed80adbc7a978f9ae2a,An apologetic Vanilla Ice has given his first ...,"Vanilla Ice, 47 - real name Robert Van Winkle ..."
287110,ffff5231e4c71544bc6c97015cdb16c60e42b3f4,America's most lethal sniper claimed he wished...,America's most lethal sniper made comment in i...
287111,ffff924b14a8d82058b6c1c5368ff1113c1632af,"By . Sara Malm . PUBLISHED: . 12:19 EST, 8 Mar...",A swarm of more than one million has crossed b...


In [7]:
# In the vanilla transformer, we can find three key components: Encoder-Decoder, the attention mechanism, and the positional encoding.
# Vanilla attention is the use of a (typically linear) combination of previous hidden vectors in the input sequence (a sentence,
# in the case of a sequence-to-sequence model).
# This combination of hidden vectors is typically concatenated with the current word's vector and fed to the next layer of the model.

# Preprocessing

In [8]:
# !pip install torch torchvision torchaudio

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math
import copy

# Input and Output Tensors for CNN/DailyMail

In [10]:
# Calculate length of each article (split by space as a rough estimate)
article_lens = train_df['article'].apply(lambda x: len(x.split()))
highlight_lens = train_df['highlights'].apply(lambda x: len(x.split()))

print(f"Avg Article Length: {article_lens.mean():.2f}")
print(f"Max Article Length: {article_lens.max()}")
print(f"Avg Highlight Length: {highlight_lens.mean():.2f}")
print(f"Max Highlight Length: {highlight_lens.max():.2f}")

Avg Article Length: 691.87
Max Article Length: 2347
Avg Highlight Length: 51.57
Max Highlight Length: 1296.00


In [11]:
max_length_src = 400
max_length_tgt = 100

In [12]:
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-base")

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [13]:
from torch.utils.data import Dataset, DataLoader

class SummarizationDataset(Dataset):
    def __init__(self, df, tokenizer, max_src_len=512, max_tgt_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_src_len = max_length_src
        self.max_tgt_len = max_length_tgt

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # We only tokenize ONE row at a time when requested
        article = str(self.df.iloc[idx]['article'])
        summary = str(self.df.iloc[idx]['highlights'])

        src_encoding = self.tokenizer(
            article, truncation=True, padding='max_length',
            max_length=self.max_src_len, return_tensors="pt"
        )
        tgt_encoding = self.tokenizer(
            summary, truncation=True, padding='max_length',
            max_length=self.max_tgt_len, return_tensors="pt"
        )

        return src_encoding['input_ids'].squeeze(), tgt_encoding['input_ids'].squeeze()


train_dataset = SummarizationDataset(train_df, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)

In [14]:
# The size of the dictionary the tokenizer uses
src_vocab_size = tokenizer.vocab_size
tgt_vocab_size = tokenizer.vocab_size

print(f"Total Vocabulary Size: {src_vocab_size}")

Total Vocabulary Size: 32000


# Building the Model

In [15]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        # Initialize dimensions
        self.d_model = d_model # Model's dimension
        self.num_heads = num_heads # Number of attention heads
        self.d_k = d_model // num_heads # Dimension of each head's key, query, and value

        # Linear layers for transforming inputs
        self.W_q = nn.Linear(d_model, d_model) # Query transformation
        self.W_k = nn.Linear(d_model, d_model) # Key transformation
        self.W_v = nn.Linear(d_model, d_model) # Value transformation
        self.W_o = nn.Linear(d_model, d_model) # Output transformation

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)

        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)

        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output

    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)

        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output

In [16]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [17]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [18]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

In [19]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

In [20]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length, device=tgt.device), diagonal=1)).bool()
        tgt_mask = tgt_mask & nopeak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)

        output = self.fc(dec_output)
        return output

# Training the Model

In [21]:
# import gc
# import torch

# # Clear Python's garbage collector
# gc.collect()

# # Clear PyTorch's cache
# torch.cuda.empty_cache()

In [22]:
src_vocab_size = tokenizer.vocab_size #1000#5000
tgt_vocab_size = tokenizer.vocab_size #5000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 400
dropout = 0.1
lr = 0.0001 #0.0001 - 0.001
#batch_size = 32, 64, 128
epochs = 3 #3-10

transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)

# Generate random sample data
# src_data = torch.randint(1, src_vocab_size, (64, max_seq_length))  # (batch_size, seq_length)
# tgt_data = torch.randint(1, tgt_vocab_size, (64, max_seq_length))  # (batch_size, seq_length)

In [23]:
# Source - https://stackoverflow.com/a/74788189
# Posted by ricksanchezdev
# Retrieved 2025-12-26, License - CC BY-SA 4.0

transformer.to(device)

Transformer(
  (encoder_embedding): Embedding(32000, 512)
  (decoder_embedding): Embedding(32000, 512)
  (positional_encoding): PositionalEncoding()
  (encoder_layers): ModuleList(
    (0-5): 6 x EncoderLayer(
      (self_attn): MultiHeadAttention(
        (W_q): Linear(in_features=512, out_features=512, bias=True)
        (W_k): Linear(in_features=512, out_features=512, bias=True)
        (W_v): Linear(in_features=512, out_features=512, bias=True)
        (W_o): Linear(in_features=512, out_features=512, bias=True)
      )
      (feed_forward): PositionWiseFeedForward(
        (fc1): Linear(in_features=512, out_features=2048, bias=True)
        (fc2): Linear(in_features=2048, out_features=512, bias=True)
        (relu): ReLU()
      )
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (decoder_layers): ModuleList(
    (0-5): 6 x DecoderLayer(

In [ ]:
# 1. Setup the "Delivery Truck"
# train_dataset = SummarizationDataset(train_df, tokenizer)
# train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# 2. Training Setup
criterion = nn.CrossEntropyLoss(ignore_index=0) # Ignores <pad> tokens in loss
optimizer = optim.Adam(transformer.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)

transformer.train()

for epoch in range(epochs):
    epoch_loss = 0

    # NEW: The inner loop that grabs 16 articles at a time
    for batch_idx, (batch_src, batch_tgt) in enumerate(train_loader):

        # Move data to GPU if you are using one
        batch_src = batch_src.to(device)
        batch_tgt = batch_tgt.to(device)

        optimizer.zero_grad()

        # Forward pass with shifted targets (Teacher Forcing)
        # batch_tgt[:, :-1] is the Decoder Input
        output = transformer(batch_src, batch_tgt[:, :-1])

        # Calculate loss against batch_tgt[:, 1:] (The Labels)
        # We flatten the tensors so CrossEntropy can compare every word's probability
        loss = criterion(output.contiguous().view(-1, tgt_vocab_size), batch_tgt[:, 1:].contiguous().view(-1))

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # Print progress every 100 batches so you know it's working
        if batch_idx % 100 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx} | Loss: {loss.item():.4f}")
        if batch_idx % 500 == 0:
            torch.save(transformer.state_dict(), f"checkpoint_batch_{batch_idx}.pth")
        # This keeps the file in Colab's local folder so you can grab it

    print(f"--- Epoch {epoch+1} Finished. Average Loss: {epoch_loss/len(train_loader):.4f} ---")

Epoch 1 | Batch 0 | Loss: 10.5199
Epoch 1 | Batch 100 | Loss: 6.7435
Epoch 1 | Batch 200 | Loss: 6.6464
Epoch 1 | Batch 300 | Loss: 6.2082
Epoch 1 | Batch 400 | Loss: 6.2899
Epoch 1 | Batch 500 | Loss: 6.2598
Epoch 1 | Batch 600 | Loss: 6.2405
Epoch 1 | Batch 700 | Loss: 6.0536
Epoch 1 | Batch 800 | Loss: 5.8833
Epoch 1 | Batch 900 | Loss: 5.8535
Epoch 1 | Batch 1000 | Loss: 5.8226
Epoch 1 | Batch 1100 | Loss: 5.8893
Epoch 1 | Batch 1200 | Loss: 5.9264
Epoch 1 | Batch 1300 | Loss: 5.5713
Epoch 1 | Batch 1400 | Loss: 5.8635
Epoch 1 | Batch 1500 | Loss: 5.7223
Epoch 1 | Batch 1600 | Loss: 5.5216
Epoch 1 | Batch 1700 | Loss: 5.5606
Epoch 1 | Batch 1800 | Loss: 5.6290
Epoch 1 | Batch 1900 | Loss: 5.6623
Epoch 1 | Batch 2000 | Loss: 5.8676
Epoch 1 | Batch 2100 | Loss: 5.6564
Epoch 1 | Batch 2200 | Loss: 5.4506
Epoch 1 | Batch 2300 | Loss: 5.4795
Epoch 1 | Batch 2400 | Loss: 5.3581
Epoch 1 | Batch 2500 | Loss: 5.4215
Epoch 1 | Batch 2600 | Loss: 5.4878
Epoch 1 | Batch 2700 | Loss: 5.1115
Epo

# Resume Training from Checkpoint

In [26]:
# 1. Setup the dataset and loader
val_dataset = SummarizationDataset(validation_df, tokenizer)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

# 2. Add this "Evaluation Function" to your script
def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch_idx, (src, tgt) in enumerate(loader):
            # Limit validation to 100 batches so it doesn't take too long
            if batch_idx > 100: break 
            
            src, tgt = src.to(device), tgt.to(device)
            output = model(src, tgt[:, :-1])
            loss = criterion(output.contiguous().view(-1, tgt_vocab_size), 
                             tgt[:, 1:].contiguous().view(-1))
            total_loss += loss.item()
            
    return total_loss / 101 # Return average loss

In [27]:
def sample_prediction(model, df, tokenizer, device, max_len=100):
    model.eval() # Switch to evaluation mode
    
    # Pick a random row from the dataframe
    random_row = df.sample(n=1).iloc[0]
    article_text = random_row['article']
    ground_truth = random_row['highlights']
    
    # Prepare input
    inputs = tokenizer(article_text, return_tensors="pt", max_length=400, truncation=True, padding='max_length')
    src = inputs["input_ids"].to(device)
    
    # Greedy decoding
    tgt_input = torch.tensor([[0]], device=device) # T5 start token is 0
    
    with torch.no_grad():
        for _ in range(max_len):
            output = model(src, tgt_input)
            next_word = output[:, -1, :].argmax(dim=-1).unsqueeze(1)
            tgt_input = torch.cat([tgt_input, next_word], dim=1)
            if next_word.item() == 1: # T5 end token is 1
                break
                
    predicted_text = tokenizer.decode(tgt_input[0], skip_special_tokens=True)
    
    print("\n" + "="*50)
    print(f"SAMPLE TEST")
    print(f"ARTICLE: {article_text[:200]}...")
    print(f"GROUND TRUTH: {ground_truth}")
    print(f"MODEL PREDICTION: {predicted_text}")
    print("="*50 + "\n")
    
    model.train() # Switch back to training mode

In [29]:
import os
import glob
import torch



# 1. Configuration for this specific resume
checkpoint_path = "/kaggle/input/vanilla-transformer/transformers/default/1/checkpoint_batch_15500.pth"
start_epoch = 1
start_batch = 15501 # We start one batch AFTER the saved one

# 2. LOAD the checkpoint
if os.path.exists(checkpoint_path):
    print(f"Found checkpoint! Loading weights from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        transformer.load_state_dict(checkpoint['model_state_dict'])
        if 'optimizer_state_dict' in checkpoint:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    else:
        transformer.load_state_dict(checkpoint)
    print(f"Resuming from Epoch {start_epoch +  1}, Batch {start_batch}")
else:
    print("CRITICAL: Checkpoint not found! Check your path.")
    # If the file is missing, we shouldn't skip batches.
    start_epoch = 0
    start_batch = 0



criterion = nn.CrossEntropyLoss(ignore_index=0) # Ignores <pad> tokens in loss
optimizer = optim.Adam(transformer.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)

transformer.train()
best_val_loss = float('inf')

for epoch in range(start_epoch, epochs):
    epoch_loss = 0
    for batch_idx, (batch_src, batch_tgt) in enumerate(train_loader):
        
        # --- SKIP LOGIC ---
        if epoch == start_epoch and batch_idx < start_batch:
            continue # Skip the math for these batches
        # ------------------

        batch_src, batch_tgt = batch_src.to(device), batch_tgt.to(device)

        optimizer.zero_grad()
        output = transformer(batch_src, batch_tgt[:, :-1])
        loss = criterion(output.contiguous().view(-1, tgt_vocab_size), 
                         batch_tgt[:, 1:].contiguous().view(-1))

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch + 1} | Batch {batch_idx} | Loss: {loss.item():.4f}")

        # 3. SMART SAVE
        if batch_idx % 500 == 0:
            # Cleanup old checkpoints in WORKING directory
            old_checkpoints = glob.glob("/kaggle/working/checkpoint_ep*_batch_*.pth")
            for f in old_checkpoints:
                try: os.remove(f)
                except: pass
            
            new_name = f"/kaggle/working/checkpoint_ep{epoch}_batch_{batch_idx}.pth"
            torch.save({
                'epoch': epoch,
                'batch': batch_idx,
                'model_state_dict': transformer.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': loss.item(),
            }, new_name)
            print(f"--> Saved new checkpoint: {new_name}")
    # After the training loop finishes for the epoch
    print(f"--- Epoch {epoch + 1} Finished. Average Loss: {epoch_loss/len(train_loader):.4f} ---")
    current_val_loss = evaluate_model(transformer, val_loader, criterion)
    print(f"Epoch {epoch + 1} | Val Loss: {current_val_loss:.4f}")
    # SAVE THE BEST MODEL ONLY
    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss
        torch.save(transformer.state_dict(), "best_model.pth")
        print("🌟 New best model saved!")
    # 2. Run the sample test
    sample_prediction(transformer, validation_df, tokenizer, device)
    
    # 3. Save your epoch checkpoint
    torch.save({
        'epoch': epoch + 1,
        'batch': batch_idx,
        'model_state_dict': transformer.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, f"checkpoint_epoch_{epoch}.pth")
    transformer.train() # Switch back to training mode for the next epoch


Found checkpoint! Loading weights from /kaggle/input/vanilla-transformer/transformers/default/1/checkpoint_batch_15500.pth...
Resuming from Epoch 2, Batch 15501
Epoch 2 | Batch 15600 | Loss: 3.6513
Epoch 2 | Batch 15700 | Loss: 3.4672
Epoch 2 | Batch 15800 | Loss: 3.8762
Epoch 2 | Batch 15900 | Loss: 3.6055
Epoch 2 | Batch 16000 | Loss: 3.4238
--> Saved new checkpoint: /kaggle/working/checkpoint_ep1_batch_16000.pth
Epoch 2 | Batch 16100 | Loss: 3.6117
Epoch 2 | Batch 16200 | Loss: 3.4437
Epoch 2 | Batch 16300 | Loss: 3.5811
Epoch 2 | Batch 16400 | Loss: 3.3985
Epoch 2 | Batch 16500 | Loss: 3.5994
--> Saved new checkpoint: /kaggle/working/checkpoint_ep1_batch_16500.pth
Epoch 2 | Batch 16600 | Loss: 3.4592
Epoch 2 | Batch 16700 | Loss: 3.5789
Epoch 2 | Batch 16800 | Loss: 3.5534
Epoch 2 | Batch 16900 | Loss: 3.7322
Epoch 2 | Batch 17000 | Loss: 3.8704
--> Saved new checkpoint: /kaggle/working/checkpoint_ep1_batch_17000.pth
Epoch 2 | Batch 17100 | Loss: 3.8146
Epoch 2 | Batch 17200 | Loss

In [30]:
import os
import glob
import torch

# 1. Define the directory and the new filename format
# Using current_epoch and current_batch variables from your loop
output_dir = "/kaggle/working"
new_filename = f"checkpoint_ep{epoch}_batch_{batch_idx}.pth"
model_path = os.path.join(output_dir, new_filename)



# 3. The Save Code (Now including Epoch and Batch for easy resuming)
torch.save({
    'epoch': epoch,
    'batch': batch_idx,
    'model_state_dict': transformer.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': {
        'src_vocab_size': src_vocab_size,
        'tgt_vocab_size': tgt_vocab_size,
        'd_model': 512,
        'num_heads': 8,
        'num_layers': 6,
        'max_seq_length': 400
    }
}, model_path)

print(f"Model saved successfully at: {model_path}")
print("You can download this from the 'Output' sidebar on the right.")

Model saved successfully at: /kaggle/working/checkpoint_ep2_batch_17944.pth
You can download this from the 'Output' sidebar on the right.


# Evaluating the Model

In [31]:
# 1. Create a smaller slice of your validation dataframe
val_df_small = validation_df.sample(n=500, random_state=42)

# 2. Use your existing class to create the dataset
val_dataset = SummarizationDataset(val_df_small, tokenizer)

# 3. Create the loader (shuffle isn't necessary for validation)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

In [32]:
# 2. Use your existing class to create the dataset
val_dataset = SummarizationDataset(validation_df, tokenizer)

# 3. Create the loader (shuffle isn't necessary for validation)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

In [33]:
transformer.eval() # Turn off dropout
total_val_loss = 0

with torch.no_grad():
    for val_src, val_tgt in val_loader:
        val_src, val_tgt = val_src.to(device), val_tgt.to(device)

        # Forward pass (Input the article and the target summary minus the last word)
        val_output = transformer(val_src, val_tgt[:, :-1])
        
        # Calculate loss against the target summary (shifted by one)
        loss = criterion(
            val_output.contiguous().view(-1, tgt_vocab_size), 
            val_tgt[:, 1:].contiguous().view(-1)
        )
        
        total_val_loss += loss.item()

avg_val_loss = total_val_loss / len(val_loader)
print(f"--- Real Validation Loss: {avg_val_loss:.4f} ---")

--- Real Validation Loss: 3.4865 ---


# Testing the Model

In [34]:
def generate_summary(model, text, tokenizer, max_len=50):
    model.eval()
    
    # 1. Tokenize the input news article
    inputs = tokenizer(text, return_tensors="pt", max_length=max_seq_length, truncation=True, padding="max_length")
    src = inputs["input_ids"].to(device)
    
    # 2. Start the decoder with the 'Start of Sentence' token
    # For T5 tokenizer, the decoder start token is usually 0 (pad/start)
    tgt_input = torch.tensor([[0]], device=device) 

    for _ in range(max_len):
        with torch.no_grad():
            # Forward pass: only through the current generated sequence
            output = model(src, tgt_input)
            
            # Get the last token's predictions and pick the highest probability
            next_word = output[:, -1, :].argmax(dim=-1).unsqueeze(1)
            
            # Append the predicted word to the input for the next step
            tgt_input = torch.cat([tgt_input, next_word], dim=1)
            
            # Stop if the model predicts the End of Sentence token (1)
            if next_word.item() == 1:
                break
    
    # 3. Convert numbers back to words
    return tokenizer.decode(tgt_input[0], skip_special_tokens=True)

In [42]:
# Pick a random article from your test set
sample_article = test_df.iloc[2]['article']
true_summary = test_df.iloc[2]['highlights']

print("--- ORIGINAL ARTICLE ---")
print(sample_article[:500] + "...") # Show first 500 chars

print("\n--- TARGET SUMMARY (Human wrote this) ---")
print(true_summary)

print("\n--- MODEL GENERATED SUMMARY ---")
generated = generate_summary(transformer, sample_article, tokenizer)
print(generated)

--- ORIGINAL ARTICLE ---
Dougie Freedman is on the verge of agreeing a new two-year deal to remain at Nottingham Forest. Freedman has stabilised Forest since he replaced cult hero Stuart Pearce and the club's owners are pleased with the job he has done at the City Ground. Dougie Freedman is set to sign a new deal at Nottingham Forest . Freedman has impressed at the City Ground since replacing Stuart Pearce in February . They made an audacious attempt on the play-off places when Freedman replaced Pearce but have tailed o...

--- TARGET SUMMARY (Human wrote this) ---
Nottingham Forest are close to extending Dougie Freedman's contract .
The Forest boss took over from former manager Stuart Pearce in February .
Freedman has since lead the club to ninth in the Championship .

--- MODEL GENERATED SUMMARY ---
ed has been a new manager since joining the club . The winger has been at the club since joining the club in January . The Argentine has not played since joining the club in the summer .


# Calculating Rouge Metrics

In [36]:
!pip install rouge

In [37]:
import json
from rouge import Rouge

# # Load some sentences
# with open('./tests/data.json') as f:
#   data = json.load(f)

# hyps, refs = map(list, zip(*[[d['hyp'], d['ref']] for d in data]))
hyps=[]
refs =[] 

sample_test_df = test_df.sample(n=50, random_state=42)

print("Generating summaries for ROUGE evaluation...")

for idx, row in sample_test_df.iterrows():
    article = row['article']
    reference = row['highlights']
    
    # Use the generate_summary function 
    prediction = generate_summary(transformer, article, tokenizer, max_len=max_length_tgt)
    
    hyps.append(prediction)
    refs.append(reference)
    
    if len(hyps) % 10 == 0:
        print(f"Processed {len(hyps)}/50 samples")
rouge = Rouge()
# scores = rouge.get_scores(hyps, refs)
# # or
scores = rouge.get_scores(hyps, refs, avg=True)

Generating summaries for ROUGE evaluation...
Processed 10/50 samples
Processed 20/50 samples
Processed 30/50 samples
Processed 40/50 samples
Processed 50/50 samples


In [39]:
print(scores)

{'rouge-1': {'r': 0.1533954263448634, 'p': 0.3075551625309958, 'f': 0.19827992375250447}, 'rouge-2': {'r': 0.02407515771019256, 'p': 0.04796131763609272, 'f': 0.030667577606746946}, 'rouge-l': {'r': 0.14340234601437074, 'p': 0.2900928778505623, 'f': 0.18594652146823099}}


In [40]:
print("\n--- ROUGE SCORES ---")
for key, value in scores.items():
    print(f"{key.upper()}:")
    print(f"  Recall:    {value['r']:.4f}")
    print(f"  Precision: {value['p']:.4f}")
    print(f"  F1-Score:  {value['f']:.4f}")


--- ROUGE SCORES ---
ROUGE-1:
  Recall:    0.1534
  Precision: 0.3076
  F1-Score:  0.1983
ROUGE-2:
  Recall:    0.0241
  Precision: 0.0480
  F1-Score:  0.0307
ROUGE-L:
  Recall:    0.1434
  Precision: 0.2901
  F1-Score:  0.1859
